In [0]:
%sql
USE flights_gold;

In [0]:
%python
from pyspark.sql.functions import avg, sum, count, round, desc, col

gold_fact = "flights_gold.fact_flights"
agg_table = "flights_gold.monthly_airline_stats"

df_fact = spark.table(gold_fact)

df_agg = df_fact.groupBy("year", "month", "airline_name").agg(
    # ilości
    count("*").alias("total_flights"),
    sum("cancelled").alias("cancelled_flights"),
    round(avg("cancelled") * 100, 2).alias("cancellation_rate"),
    
    # opóźnienia
    round(avg("departure_delay"), 2).alias("avg_dep_delay"),
    round(avg("arrival_delay"), 2).alias("avg_arr_delay"),
    
    # czasy trwania
    round(avg("scheduled_time"), 2).alias("avg_scheduled_time"),
    round(avg("elapsed_time"), 2).alias("avg_elapsed_time"),
    round(avg("air_time"), 2).alias("avg_air_time"),
    # ---------------------------

    # kołowanie
    round(avg("taxi_out"), 2).alias("avg_taxi_out"),
    round(avg("taxi_in"), 2).alias("avg_taxi_in"),
    
    # przyczyny opóźnień
    round(avg("weather_delay"), 2).alias("avg_weather_delay"),
    round(avg("airline_delay"), 2).alias("avg_airline_delay"),
    round(avg("security_delay"), 2).alias("avg_security_delay"),
    round(avg("air_system_delay"), 2).alias("avg_system_delay"),
    round(avg("late_aircraft_delay"), 2).alias("avg_late_aircraft_delay")

).orderBy("year", "month", desc("avg_dep_delay"))

df_agg.write.format("delta").mode("overwrite").saveAsTable(agg_table)

In [0]:
%python
display(df_agg.limit(5))

year,month,airline_name,total_flights,cancelled_flights,cancellation_rate,avg_dep_delay,avg_arr_delay,avg_scheduled_time,avg_elapsed_time,avg_air_time,avg_taxi_out,avg_taxi_in,avg_weather_delay,avg_airline_delay,avg_security_delay,avg_system_delay,avg_late_aircraft_delay
2015,1,Frontier Airlines Inc.,6829,89,1.3,17.98,18.36,153.04,153.47,129.3,15.76,8.4,0.19,4.87,0.0,7.09,9.76
2015,1,American Eagle Airlines Inc.,29900,2288,7.65,16.08,18.16,94.85,97.17,69.03,17.56,10.59,1.81,4.93,0.03,5.09,8.25
2015,1,United Air Lines Inc.,38395,967,2.52,14.01,6.35,200.29,192.34,165.93,17.52,8.9,0.86,4.52,0.0,2.97,4.36
2015,1,Spirit Air Lines,8743,98,1.12,13.15,11.4,160.86,159.28,134.93,14.85,9.5,0.07,2.32,0.04,10.74,2.82
2015,1,Skywest Airlines Inc.,48114,1262,2.62,12.16,10.89,101.27,100.26,74.62,18.49,7.17,0.64,4.48,0.01,2.84,7.25
